# ENTSO-E Data Cleaning

### Poland (PL)

**Generation:** Remove `Other Renewable`, `Other`, `Hydro Pumped Storage – Actual Consumption`, and `Wind Offshore`.

**Wind/Solar Forecast:** Remove `Wind Offshore`.

**Day-ahead Prices:** Deduplicate timestamps and keep the first occurrence.

**Unavailability:** Drop `generation_units_unavailability` and `production_units_unavailability`. Most records were published only after the outage ended, and usable history starts in autumn 2025.

### Germany / Luxembourg (DE_LU)

**Unavailability:** Drop `generation_units_unavailability` and `production_units_unavailability`, as for PL.

### France (FR)

**Wind/Solar Forecast:** Remove `Wind Offshore`.

**Unavailability:** Drop `generation_units_unavailability` and `production_units_unavailability`, as for PL.

### Lithuania (LT)

**Generation Forecast:** Exclude `generation_forecast` entirely due to insufficient data quality.

In [1]:
import pandas as pd
import logging
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent.parent))

from src.config import (
    DATA_DIR,
    DOWNLOAD_START_DATE,
    DOWNLOAD_END_DATE,
    ENTSOE_BIDDING_ZONE_CODES,
)
from src.utils.data_quality import print_missing_values, print_index_summary

from src.utils.cleaning import (
    clean_pl,
    clean_de_lu,
    clean_fr,
    clean_lt,
)
from utils.file_io import inspect_entsoe_files, save_entsoe_data, load_entsoe_data

In [2]:
date_range = f"{DOWNLOAD_START_DATE}_{DOWNLOAD_END_DATE}"
logging.basicConfig(level=logging.INFO)

In [3]:
data_entsoe = load_entsoe_data(
    DATA_DIR / "raw" / "entsoe",
    date_range,
)

In [4]:
data_entsoe["PL"] = clean_pl(data_entsoe["PL"])
data_entsoe["DE_LU"] = clean_de_lu(data_entsoe["DE_LU"])
data_entsoe["FR"] = clean_fr(data_entsoe["FR"])
data_entsoe["LT"] = clean_lt(data_entsoe["LT"])

codes = ["PL", "DE_LU", "FR", "LT"]

## 1. Index Integrity – Confirmation

The cleaned datasets no longer contain duplicated index values: the 45 exact duplicates in PL `day_ahead_prices` are gone. The only repeated timestamps left come from the Summer/Winter time switch. Gaps in the time index are not part of this step and are handled later, during preprocessing.

In [5]:
for code in codes:
    print(f"===== {code} {"=" * 20}")
    for n, d in data_entsoe[code].items():
        print_index_summary(d, n)

===== PL ====================

--- [load] Index summary ---
Duplicates: 0
Duplicates due to Summer/Winter time switch: 9
Timezone:   Europe/Warsaw

Time steps:
  0 days 00:15:00: 79191
  0 days 01:00:00: 13441

--- [generation] Index summary ---
Duplicates: 0
Duplicates due to Summer/Winter time switch: 9
Timezone:   Europe/Warsaw

Time steps:
  0 days 00:15:00: 79191
  0 days 01:00:00: 13441

--- [load_forecast] Index summary ---
Duplicates: 0
Duplicates due to Summer/Winter time switch: 9
Timezone:   Europe/Warsaw

Time steps:
  0 days 00:15:00: 79191
  0 days 01:00:00: 13441

--- [day_ahead_prices] Index summary ---
Duplicates: 0
Duplicates due to Summer/Winter time switch: 6
Timezone:   Europe/Warsaw

Time steps:
  0 days 00:15:00: 33600
  0 days 01:00:00: 24839

--- [generation_wind_solar_forecast] Index summary ---
Duplicates: 0
Duplicates due to Summer/Winter time switch: 9
Timezone:   Europe/Warsaw

Time steps:
  0 days 00:15:00: 79191
  0 days 01:00:00: 13441

--- [generation_

## 2. Missing Values – Confirmation

No missing values remain in PL or LT, as the affected columns (e.g. `Wind Offshore`) have been removed. The only exceptions are `Solar` and `Wind Onshore` in the DE_LU (12 values) and FR (27 values) `generation_wind_solar_forecast`, which are DST-related and short gaps to be filled during preprocessing.

In [6]:
for code in codes:
    print(f"===== {code} {"=" * 20}")
    for n, d in data_entsoe[code].items():
        print_missing_values(d, n)

===== PL ====================

--- [load] Missing values ---
No missing values.

--- [generation] Missing values ---
No missing values.

--- [load_forecast] Missing values ---
No missing values.

--- [day_ahead_prices] Missing values ---
No missing values.

--- [generation_wind_solar_forecast] Missing values ---
No missing values.

--- [generation_forecast] Missing values ---
No missing values.
===== DE_LU ====================

--- [load_forecast] Missing values ---
No missing values.

--- [day_ahead_prices] Missing values ---
No missing values.

--- [generation_wind_solar_forecast] Missing values ---
Solar           12
Wind Onshore    12

--- [generation_forecast] Missing values ---
No missing values.
===== FR ====================

--- [load_forecast] Missing values ---
No missing values.

--- [day_ahead_prices] Missing values ---
No missing values.

--- [generation_wind_solar_forecast] Missing values ---
Solar           27
Wind Onshore    27

--- [generation_forecast] Missing values 

## 3. Conclusions

The cleaning decisions from the data quality audit have been applied to PL, DE_LU, FR and LT, and the checks above confirm the result. The cleaned datasets of all bidding zones, including those that required no cleaning, are saved to `data/clean/entsoe/` and verified by reading the files back.

## 4. Saving Clean Data

In [7]:
save_entsoe_data(
    data_entsoe,
    DATA_DIR / "clean" / "entsoe",
    date_range,
)

INFO:utils.file_io:Saved clean/entsoe/AT/2022-12-01_2026-09-16/load_forecast.parquet
INFO:utils.file_io:Saved clean/entsoe/AT/2022-12-01_2026-09-16/water_reservoirs_hydro_storage.parquet
INFO:utils.file_io:Saved clean/entsoe/AT/2022-12-01_2026-09-16/day_ahead_prices.parquet
INFO:utils.file_io:Saved clean/entsoe/AT/2022-12-01_2026-09-16/generation_wind_solar_forecast.parquet
INFO:utils.file_io:Saved clean/entsoe/AT/2022-12-01_2026-09-16/generation_forecast.parquet
INFO:utils.file_io:Saved clean/entsoe/CZ/2022-12-01_2026-09-16/load_forecast.parquet
INFO:utils.file_io:Saved clean/entsoe/CZ/2022-12-01_2026-09-16/day_ahead_prices.parquet
INFO:utils.file_io:Saved clean/entsoe/CZ/2022-12-01_2026-09-16/generation_wind_solar_forecast.parquet
INFO:utils.file_io:Saved clean/entsoe/CZ/2022-12-01_2026-09-16/generation_forecast.parquet
INFO:utils.file_io:Saved clean/entsoe/DE_LU/2022-12-01_2026-09-16/load_forecast.parquet
INFO:utils.file_io:Saved clean/entsoe/DE_LU/2022-12-01_2026-09-16/day_ahead_pr

In [8]:
inspect_entsoe_files(
    DATA_DIR / "clean" / "entsoe",
    date_range,
)


--- AT (5 file/s) ------------------------------
  [FILE] data/clean/entsoe/AT/2022-12-01_2026-09-16/load_forecast.parquet | shape: (132956, 1) | idx: 2022-12-01 00:00:00+01:00-2026-09-15 23:45:00+02:00
  [FILE] data/clean/entsoe/AT/2022-12-01_2026-09-16/water_reservoirs_hydro_storage.parquet | shape: (196, 1) | idx: 2022-11-27 23:00:00+00:00-2026-08-30 23:00:00+00:00
  [FILE] data/clean/entsoe/AT/2022-12-01_2026-09-16/day_ahead_prices.parquet | shape: (58437, 1) | idx: 2022-12-01 00:00:00+01:00-2026-09-16 00:00:00+02:00
  [FILE] data/clean/entsoe/AT/2022-12-01_2026-09-16/generation_wind_solar_forecast.parquet | shape: (132956, 2) | idx: 2022-12-01 00:00:00+01:00-2026-09-15 23:45:00+02:00
  [FILE] data/clean/entsoe/AT/2022-12-01_2026-09-16/generation_forecast.parquet | shape: (54836, 2) | idx: 2022-12-01 00:00:00+01:00-2026-09-15 23:45:00+02:00

--- CZ (4 file/s) ------------------------------
  [FILE] data/clean/entsoe/CZ/2022-12-01_2026-09-16/load_forecast.parquet | shape: (91409, 1